# JEV-Bench parallel reformulation

Import the new data-only generator without changing model code. Choose the save directory and number of new rows.


In [ ]:
from pathlib import Path
import sys, json
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "jevbench").exists(): PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from reformulate_dataset import build


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "jev_bench_processed_reformulated"
NUM_GENERATED = 2000  # exact total number of NEW examples
WORKERS = 4
SEED = 42
CONFIGS = None  # load ALL dataset configs; e.g. ["mnli", "mmlu"] for a trial
SPLITS = ("train", "validation", "test")


In [ ]:
saved_dir = build(
    output_dir=OUTPUT_DIR,
    configs=CONFIGS,
    splits=SPLITS,
    num_generated=NUM_GENERATED,
    workers=WORKERS,
    seed=SEED,
    save_raw=False,
    force=False,
)
print(saved_dir)


In [ ]:
manifest = json.loads((saved_dir / "manifest.json").read_text(encoding="utf-8"))
print("Originals:", manifest["original_split_counts"])
print("New:", manifest["generated_split_counts"])
print("Requested and generated:", {k:(v["requested"],v["generated"]) for k,v in manifest["generation_reports"].items()})


In [ ]:
def sample_rows(path, n=4):
    with path.open(encoding="utf-8") as f:
        for i,line in enumerate(f):
            if i>=n: break
            yield json.loads(line)
for row in sample_rows(saved_dir / "train.jsonl"):
    print(row["source_config"], "original:", row["original"], "strategy:", (row["generation"] or {}).get("strategy"))
